# Setup

In [14]:
!pip install opik pandas


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [15]:
import pandas as pd
import json
import os
import re
from opik import Opik
from opik.evaluation.metrics import base_metric, score_result
from opik.integrations.openai import track_openai
from opik.evaluation import evaluate
from openai import OpenAI
from typing import Any

# Opik Configuration
os.environ["OPIK_URL_OVERRIDE"] = "https://3.110.54.210/api"
os.environ["OPIK_CHECK_TLS_CERTIFICATE"] = "false"
if not os.environ.get("OPENROUTER_API_KEY"):
    raise ValueError("Set OPENROUTER_API_KEY in your environment before running this notebook")
os.environ["OPIK_PROJECT_NAME"] = "AI Evaluations"

# Experiment Configuration
EXPERIMENT_NAME = "textual-eval_google/openai/gpt-5.4-mini"
MODEL_NAME = "openai/gpt-5.4-mini"
DATASET_NAME = "ai_eval_textual_prod_10"

# OpenRouter client
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

# Opik client
opik_client = Opik(project_name=os.environ["OPIK_PROJECT_NAME"])

print("✅ Environment configured")

✅ Environment configured


# Upload Dataset

In [16]:
dataset = opik_client.get_or_create_dataset(DATASET_NAME)
print(dataset)

items = dataset.get_items()
current_size = len(items)

if current_size == 0:
    df = pd.read_csv("your_dataset.csv")
    dataset.insert_from_pandas(dataframe=df)
    print(f"✅ Uploaded {len(df)} rows to '{DATASET_NAME}'")
else:
    print(f"✅ Dataset '{DATASET_NAME}' already exists with {current_size} rows, skipping upload")

✅ Dataset 'ai_eval_textual_prod_10' already exists with 10 rows, skipping upload


# Set Up Model Config

In [17]:
# ── All metrics use this shared evaluator LLM ──────────────────────────────
EVALUATOR_MODEL = "google/gemini-2.5-pro"

def call_evaluator(prompt: str) -> dict:
    response = openrouter_client.chat.completions.create(
        model=EVALUATOR_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.001,
    )
    raw = response.choices[0].message.content
    if not raw:
        raise ValueError("Evaluator LLM returned empty content")
    raw = raw.strip()
    raw = re.sub(r"```json|```", "", raw).strip()
    return json.loads(raw)

print(f"✅ Evaluator LLM configured: {EVALUATOR_MODEL}")

✅ Evaluator LLM configured: google/gemini-2.5-pro


# Create your custom evaluators

In [18]:
def _safe_str(value) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and pd.isna(value):
        return ""
    return str(value)


def build_textual_eval_prompt(dataset_item: dict) -> tuple[str, str]:
    """Build system + user prompts for the textual evaluation LLM call."""
    question_text = _safe_str(dataset_item.get("question_text"))
    user_answer = _safe_str(dataset_item.get("user_answer"))

    evaluation_objective = _safe_str(dataset_item.get("evaluation_objective"))
    evaluation_objective = evaluation_objective.replace("{question_text}", question_text)
    evaluation_objective = evaluation_objective.replace("{user_answer}", user_answer)

    user_sections = [evaluation_objective]

    general_criteria = _safe_str(dataset_item.get("general_evaluation_criteria"))
    if general_criteria:
        user_sections.append(general_criteria)

    specific_criteria = _safe_str(dataset_item.get("specific_evaluation_criteria"))
    if specific_criteria:
        user_sections.append(f"Specific Evaluation Criteria:\n{specific_criteria}")

    critical_cases = _safe_str(dataset_item.get("critical_cases"))
    if critical_cases:
        user_sections.append(f"Critical Cases:\n{critical_cases}")

    feedback_guidelines = _safe_str(dataset_item.get("feedback_guidelines"))
    if feedback_guidelines:
        user_sections.append(f"Feedback Guidelines:\n{feedback_guidelines}")

    response_format = _safe_str(dataset_item.get("response_format"))
    if response_format:
        user_sections.append(response_format)

    system_prompt = _safe_str(dataset_item.get("meta_role"))
    user_prompt = "\n\n".join(user_sections)
    return system_prompt, user_prompt


print("✅ Textual evaluation helpers defined")

✅ Textual evaluation helpers defined


# GeneralCriteriaEvaluationMetric

In [19]:
class GeneralCriteriaEvaluationMetric(base_metric.BaseMetric):
    """
    Scores 1 if evaluator output strictly follows general_evaluation_criteria.
    Scores 0 on any deviation, omission, or contradiction.
    """
    def __init__(self, name: str = "general_criteria_evaluation"):
        self.name = name
        self.prompt_template = """
You are an automated auditor checking Evaluation Criteria Fidelity — whether the evaluator's output correctly follows the general_evaluation_criteria declared in the evaluation prompt.

Inputs:
question: {question}
student_answer: {student_answer}
evaluation_prompt:
{{
  "general_evaluation_criteria": "{general_evaluation_criteria}"
}}
evaluator_output: {output}

Evaluation Criteria:
Check if:
- The evaluation logic clearly implements the declared rules in evaluation_criteria.
- The scoring or decision in the evaluator output reflects the conditions mentioned in the general evaluation criteria (e.g., penalty for grammar issues, scale adherence, irrelevance handling).
- No criteria point is ignored (e.g., evaluator skips relevance check when the rule mandates it).

Scoring Method:
Score = 1 → The evaluator strictly followed every rule stated in evaluation_criteria.
Score = 0 → Any deviation, omission, or contradiction between declared criteria and evaluator's behavior.

Be extremely strict.

Examples:
✅ Score: 1 — Criteria says: "If grammar incorrect → score = 0." Evaluator gives score = 0 for wrong grammar.
❌ Score: 0 — Criteria says: "If irrelevant → score = 0." Evaluator gives score = 80 to an irrelevant answer.

Return as JSON:
{{
  "reason": "<specific explanation>",
  "score": <0 or 1>
}}
Return only valid JSON with exactly two keys: reason and score.
"""

    def score(
        self,
        output: str,
        question: str = "",
        student_answer: str = "",
        general_evaluation_criteria: str = "",
        **ignored_kwargs: Any,
    ):
        if not general_evaluation_criteria:
            return score_result.ScoreResult(
                name=self.name,
                value=1.0,
                reason="No general_evaluation_criteria declared for this item; metric not applicable.",
            )
        prompt = self.prompt_template.format(
            question=_safe_str(question),
            student_answer=_safe_str(student_answer),
            general_evaluation_criteria=_safe_str(general_evaluation_criteria),
            output=output,
        )
        result = call_evaluator(prompt)
        return score_result.ScoreResult(
            name=self.name,
            value=float(result["score"]),
            reason=result["reason"],
        )

print("✅ GeneralCriteriaEvaluationMetric defined")

✅ GeneralCriteriaEvaluationMetric defined


# SpecificCriteriaEvaluationMetric

In [20]:
class SpecificCriteriaEvaluationMetric(base_metric.BaseMetric):
    """
    Scores 1 if evaluator output strictly follows specific_evaluation_criteria.
    Scores 0 on any deviation, omission, or contradiction.
    """
    def __init__(self, name: str = "specific_criteria_evaluation"):
        self.name = name
        self.prompt_template = """
You are an automated auditor checking Evaluation Criteria Fidelity — whether the evaluator's output correctly follows the specific_evaluation_criteria declared in the evaluation prompt.

Inputs:
question: {question}
student_answer: {student_answer}
evaluation_prompt:
{{
  "specific_evaluation_criteria": "{specific_evaluation_criteria}"
}}
evaluator_output: {output}

Evaluation Criteria:
Check if:
- The evaluation logic clearly implements the declared rules in specific_evaluation_criteria.
- The scoring or decision in the evaluator output reflects those specific criteria conditions (e.g., penalty for grammar issues, scale adherence, irrelevance handling).
- No criteria point is ignored (e.g., evaluator skips relevance check when the rule mandates it).

Scoring Method:
Score = 1 → The evaluator strictly followed every rule stated in specific_evaluation_criteria.
Score = 0 → Any deviation, omission, or contradiction between declared criteria and evaluator's behavior.

Be extremely strict.

Examples:
✅ Score: 1 — Criteria says: "If grammar incorrect → score = 0." Evaluator gives score = 0 for wrong grammar.
❌ Score: 0 — Criteria says: "If irrelevant → score = 0." Evaluator gives score = 80 to an irrelevant answer.

Return as JSON:
{{
  "reason": "<specific explanation>",
  "score": <0 or 1>
}}
Return only valid JSON with exactly two keys: reason and score.
"""

    def score(
        self,
        output: str,
        question: str = "",
        student_answer: str = "",
        specific_evaluation_criteria: str = "",
        **ignored_kwargs: Any,
    ):
        if not specific_evaluation_criteria:
            return score_result.ScoreResult(
                name=self.name,
                value=1.0,
                reason="No specific_evaluation_criteria declared for this item; metric not applicable.",
            )
        prompt = self.prompt_template.format(
            question=_safe_str(question),
            student_answer=_safe_str(student_answer),
            specific_evaluation_criteria=_safe_str(specific_evaluation_criteria),
            output=output,
        )
        result = call_evaluator(prompt)
        return score_result.ScoreResult(
            name=self.name,
            value=float(result["score"]),
            reason=result["reason"],
        )

print("✅ SpecificCriteriaEvaluationMetric defined")

✅ SpecificCriteriaEvaluationMetric defined


# CriticalCasesEvaluationMetric

In [21]:
class CriticalCasesEvaluationMetric(base_metric.BaseMetric):
    """
    Scores 1 if evaluator output strictly follows critical_cases rules.
    Scores 0 on any deviation, omission, or contradiction.
    """
    def __init__(self, name: str = "critical_cases_evaluation"):
        self.name = name
        self.prompt_template = """
You are an automated auditor checking Evaluation Criteria Fidelity — whether the evaluator's output correctly follows the critical_cases declared in the evaluation prompt.

Inputs:
question: {question}
student_answer: {student_answer}
evaluation_prompt:
{{
  "critical_cases": "{critical_cases}"
}}
evaluator_output: {output}

Evaluation Criteria:
Check if:
- The evaluation logic clearly implements the declared rules in critical_cases.
- The scoring or decision in the evaluator output reflects those critical conditions (e.g., penalty for grammar issues, scale adherence, irrelevance handling).
- No critical point is ignored (e.g., evaluator skips relevance check when the rule mandates it).

Scoring Method:
Score = 1 → The evaluator strictly followed every rule stated in critical_cases.
Score = 0 → Any deviation, omission, or contradiction between declared cases and evaluator's behavior.

Be extremely strict.

Examples:
✅ Score: 1 — Criteria says: "If grammar incorrect → score = 0." Evaluator gives score = 0 for wrong grammar.
❌ Score: 0 — Criteria says: "If irrelevant → score = 0." Evaluator gives score = 80 to an irrelevant answer.

Return as JSON:
{{
  "reason": "<specific explanation>",
  "score": <0 or 1>
}}
Return only valid JSON with exactly two keys: reason and score.
"""

    def score(
        self,
        output: str,
        question: str = "",
        student_answer: str = "",
        critical_cases: str = "",
        **ignored_kwargs: Any,
    ):
        if not critical_cases:
            return score_result.ScoreResult(
                name=self.name,
                value=1.0,
                reason="No critical_cases declared for this item; metric not applicable.",
            )
        prompt = self.prompt_template.format(
            question=_safe_str(question),
            student_answer=_safe_str(student_answer),
            critical_cases=_safe_str(critical_cases),
            output=output,
        )
        result = call_evaluator(prompt)
        return score_result.ScoreResult(
            name=self.name,
            value=float(result["score"]),
            reason=result["reason"],
        )

print("✅ CriticalCasesEvaluationMetric defined")

✅ CriticalCasesEvaluationMetric defined


# ScoringConsistencyMetric

In [22]:
class ScoringConsistencyMetric(base_metric.BaseMetric):
    """
    Scores 1 if the numeric score aligns with feedback justification.
    Scores 0 on any mismatch between justification and numeric score.
    """
    def __init__(self, name: str = "scoring_consistency"):
        self.name = name
        self.prompt_template = """
You are evaluating whether the numerical or categorical score in the evaluator's output is consistent with the reasoning and criteria.

Inputs:
question: {question}
student_answer: {student_answer}
evaluator_output: {output}

Evaluation Criteria:
- The assigned score (e.g., similarity_percentage, grade, or pass/fail) must logically align with the explanation.
- High score ↔ high quality feedback and correct answer.
- Low score ↔ presence of flaws clearly justified in feedback.
- No random or contradictory scoring.

Scoring Method:
Score = 1 → Score value and justification match declared criteria.
Score = 0 → Any mismatch between justification and numeric score.

Examples:
✅ Score: 1 — Evaluator: "Grammar incorrect → similarity 20." Clear justification and alignment.
❌ Score: 0 — Evaluator: "Answer wrong → similarity 90." Contradiction.

Return as JSON:
{{
  "reason": "<specific explanation>",
  "score": <0 or 1>
}}
Return only valid JSON with exactly two keys: reason and score.
"""

    def score(
        self,
        output: str,
        question: str = "",
        student_answer: str = "",
        **ignored_kwargs: Any,
    ):
        prompt = self.prompt_template.format(
            question=_safe_str(question),
            student_answer=_safe_str(student_answer),
            output=output,
        )
        result = call_evaluator(prompt)
        return score_result.ScoreResult(
            name=self.name,
            value=float(result["score"]),
            reason=result["reason"],
        )

print("✅ ScoringConsistencyMetric defined")

✅ ScoringConsistencyMetric defined


# FeedbackRelevanceMetric

In [23]:
class FeedbackRelevanceMetric(base_metric.BaseMetric):
    """
    Scores 1 if feedback fully follows declared feedback_guidelines.
    Scores 0 on any deviation or irrelevant feedback.
    """
    def __init__(self, name: str = "feedback_relevance"):
        self.name = name
        self.prompt_template = """
You are checking if the feedback portion of the evaluator's output is relevant, actionable, and aligned with feedback_guidelines.

Inputs:
question: {question}
student_answer: {student_answer}
evaluation_prompt:
{{
  "feedback_guidelines": "{feedback_guidelines}"
}}
evaluator_output: {output}

Evaluation Criteria:
- Feedback should adhere to tone and scope in feedback_guidelines.
- Should mention only improvements relevant to the question/answer.
- Should avoid disallowed content (e.g., off-topic suggestions, external resources).
- If answer is correct, feedback must be short and affirming — not overexplaining.

Scoring Method:
Score = 1 → Feedback fully follows the declared feedback_guidelines.
Score = 0 → Any deviation or irrelevant feedback.

Examples:
✅ Score: 1 — "Good job! Just check punctuation in your next attempt." Matches guideline, polite, relevant.
❌ Score: 0 — "You should practice listening better." Off-topic; violates feedback guideline.

Return as JSON:
{{
  "reason": "<specific explanation>",
  "score": <0 or 1>
}}
Return only valid JSON with exactly two keys: reason and score.
"""

    def score(
        self,
        output: str,
        question: str = "",
        student_answer: str = "",
        feedback_guidelines: str = "",
        **ignored_kwargs: Any,
    ):
        if not feedback_guidelines:
            return score_result.ScoreResult(
                name=self.name,
                value=1.0,
                reason="No feedback_guidelines declared for this item; metric not applicable.",
            )
        prompt = self.prompt_template.format(
            question=_safe_str(question),
            student_answer=_safe_str(student_answer),
            feedback_guidelines=_safe_str(feedback_guidelines),
            output=output,
        )
        result = call_evaluator(prompt)
        return score_result.ScoreResult(
            name=self.name,
            value=float(result["score"]),
            reason=result["reason"],
        )

print("✅ FeedbackRelevanceMetric defined")

✅ FeedbackRelevanceMetric defined


# StructuralComplianceMetric

In [24]:
class StructuralComplianceMetric(base_metric.BaseMetric):
    """
    Scores 1 if evaluator output matches declared response_format exactly.
    Scores 0 on any JSON or key mismatch.
    """
    def __init__(self, name: str = "structural_compliance"):
        self.name = name
        self.prompt_template = """
You are checking if the final evaluator output follows the declared response_format exactly (e.g., valid JSON, required fields only).

Inputs:
question: {question}
student_answer: {student_answer}
evaluation_prompt:
{{
  "response_format": "{response_format}"
}}
evaluator_output: {output}

Evaluation Criteria:
- Response must be valid RFC8259 JSON.
- Must contain exactly the required keys — no extras or omissions.
- No invalid enclosing characters, arrays, or extra text.
- Should be parsable via json.loads() without error.

Scoring Method:
Score = 1 → Format fully matches declared specification.
Score = 0 → Any JSON or key mismatch.

Examples:
✅ Score: 1 — Evaluator output matches declared format perfectly.
❌ Score: 0 — Evaluator adds extra key or returns text outside JSON.

Return as JSON:
{{
  "reason": "<specific explanation>",
  "score": <0 or 1>
}}
Return only valid JSON with exactly two keys: reason and score.
"""

    def score(
        self,
        output: str,
        question: str = "",
        student_answer: str = "",
        response_format: str = "",
        **ignored_kwargs: Any,
    ):
        if not response_format:
            return score_result.ScoreResult(
                name=self.name,
                value=1.0,
                reason="No response_format declared for this item; metric not applicable.",
            )
        prompt = self.prompt_template.format(
            question=_safe_str(question),
            student_answer=_safe_str(student_answer),
            response_format=_safe_str(response_format),
            output=output,
        )
        result = call_evaluator(prompt)
        return score_result.ScoreResult(
            name=self.name,
            value=float(result["score"]),
            reason=result["reason"],
        )

print("✅ StructuralComplianceMetric defined")

✅ StructuralComplianceMetric defined


# Assemble Metrics

In [25]:
textual_eval_metrics = [
    GeneralCriteriaEvaluationMetric(),
    SpecificCriteriaEvaluationMetric(),
    CriticalCasesEvaluationMetric(),
    ScoringConsistencyMetric(),
    FeedbackRelevanceMetric(),
    StructuralComplianceMetric(),
]

print(f"✅ {len(textual_eval_metrics)} metrics ready:")
for m in textual_eval_metrics:
    print(f"   • {m.name}")

✅ 6 metrics ready:
   • general_criteria_evaluation
   • specific_criteria_evaluation
   • critical_cases_evaluation
   • scoring_consistency
   • feedback_relevance
   • structural_compliance


# Experiment — Run Evaluation

In [26]:
dataset = opik_client.get_dataset(name=DATASET_NAME)
tracked_llm_client = track_openai(openrouter_client)


def llm_call(dataset_item: dict) -> str:
    try:
        system_prompt, user_prompt = build_textual_eval_prompt(dataset_item)
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": user_prompt})
        response = tracked_llm_client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            temperature=0.001,
        )
        content = response.choices[0].message.content
        return content if content else ""
    except Exception as e:
        return f"Error: {str(e)}"


def evaluation_task(dataset_item: dict) -> dict:
    output = llm_call(dataset_item)
    return {
        "output": output,
        "question": _safe_str(dataset_item.get("question_text")),
        "student_answer": _safe_str(dataset_item.get("user_answer")),
        "general_evaluation_criteria": _safe_str(dataset_item.get("general_evaluation_criteria")),
        "specific_evaluation_criteria": _safe_str(dataset_item.get("specific_evaluation_criteria")),
        "critical_cases": _safe_str(dataset_item.get("critical_cases")),
        "feedback_guidelines": _safe_str(dataset_item.get("feedback_guidelines")),
        "response_format": _safe_str(dataset_item.get("response_format")),
    }


print(f"🚀 Starting evaluation : {EXPERIMENT_NAME}")
print(f"📁 Dataset             : {DATASET_NAME}")
print(f"🤖 Model               : {MODEL_NAME}")
print(f"📊 Metrics             : {len(textual_eval_metrics)}")
print(f"Evaluate object : {evaluate}")

dataset.get_version_info = lambda: None

eval_results = evaluate(
    experiment_name=EXPERIMENT_NAME,
    dataset=dataset,
    task=evaluation_task,
    scoring_metrics=textual_eval_metrics,
    task_threads=2,
    verbose=1,
)

print("✅ Evaluation completed")

🚀 Starting evaluation : textual-eval_google/openai/gpt-5.4-mini
📁 Dataset             : ai_eval_textual_prod_10
🤖 Model               : openai/gpt-5.4-mini
📊 Metrics             : 6
Evaluate object : <function evaluate at 0x767eae6e07c0>


Evaluation (there might be a delay before the first items are processed): 0it [00:00, ?it/s]

╭─ ai_eval_textual_prod_10 (10 samples) ─────╮
│                                            │
│ Total time:        00:06:25                │
│ Number of samples: 10                      │
│                                            │
│ general_criteria_evaluation: 0.8000 (avg)  │
│ specific_criteria_evaluation: 0.3000 (avg) │
│ critical_cases_evaluation: 0.3000 (avg)    │
│ scoring_consistency: 0.8000 (avg)          │
│ feedback_relevance: 0.3000 (avg)           │
│ structural_compliance: 1.0000 (avg)        │
│                                            │
╰────────────────────────────────────────────╯

Uploading results to Opik ...

View the results ]8;id=16058317;https://3.110.54.210/api/v1/session/redirect/experiments/?experiment_id=019f03cf-6074-7759-b843-6d63e1b510e3&dataset_id=019a729b-d236-7100-a355-e0474ae1da85&path=aHR0cHM6Ly8zLjExMC41NC4yMTAvYXBp\in your Opik dashboard]8;;\.

✅ Evaluation completed
